In [1]:
from utils.spark_session import get_sparkSession 


In [2]:
spark = get_sparkSession("Generate athena DDLs")

In [3]:
print("SPARK-APP : Spark UI - " + spark.sparkContext.uiWebUrl)

SPARK-APP : Spark UI - http://ba7b2bbb9aec:4040


In [4]:
_schema_name = "gold"
_symlink_path = "s3a://ecommerce-pooja/pipeline/warehouse"
_script_name = "generate_athena_ddl.ipynb"


In [5]:
db_tables = spark.sql(f"show tables in {_schema_name}").collect()

In [6]:
for row in db_tables:
    
    _table = row["tableName"]
    
    df = spark.read.table(f"{_schema_name}.{_table}")
    
    columns = []
    for col_name, col_type in df.dtypes :
            columns.append(f"{col_name} {col_type}")
        
    print(f"CREATE EXTERNAL TABLE IF NOT EXISTS {_schema_name}.{_table} (")
    print(", \n".join(columns))
    print(")")
    print(f"""
    ROW FORMAT SERDE 'org.apache.hadoop.hive.ql.io.parquet.serde.ParquetHiveSerDe'
    STORED AS INPUTFORMAT 'org.apache.hadoop.hive.ql.io.SymlinkTextInputFormat'
    OUTPUTFORMAT 'org.apache.hadoop.hive.ql.io.HiveIgnoreKeyTextOutputFormat'
    LOCATION '{_symlink_path}/{_schema_name}.db/{_table}/'
    TBLPROPERTIES ('table_type'='SYMLINK');
    """)
    

CREATE EXTERNAL TABLE IF NOT EXISTS gold.dim_channel (
channel_sk int, 
channel_code string, 
channel_name string, 
channel_type string, 
channel_desc string, 
is_deleted boolean, 
created_on timestamp, 
created_by string, 
modified_on timestamp, 
modified_by string
)

    ROW FORMAT SERDE 'org.apache.hadoop.hive.ql.io.parquet.serde.ParquetHiveSerDe'
    STORED AS INPUTFORMAT 'org.apache.hadoop.hive.ql.io.SymlinkTextInputFormat'
    OUTPUTFORMAT 'org.apache.hadoop.hive.ql.io.HiveIgnoreKeyTextOutputFormat'
    LOCATION 's3a://ecommerce-pooja/pipeline/warehouse/gold.db/dim_channel/'
    TBLPROPERTIES ('table_type'='SYMLINK');
    
CREATE EXTERNAL TABLE IF NOT EXISTS gold.dim_currency (
currency_sk int, 
currency_code string, 
currency_name string, 
currency_country string, 
is_deleted boolean, 
created_on timestamp, 
created_by string, 
modified_on timestamp, 
modified_by string
)

    ROW FORMAT SERDE 'org.apache.hadoop.hive.ql.io.parquet.serde.ParquetHiveSerDe'
    STORED AS INPUTFORMA

In [ ]:
spark.stop()